# Evacuation Simulation Runner and Analysis

This notebook lets you:

1. Choose and run a simulation case from your YAML configs.
2. Inspect the generated run directory and artifacts.
3. Explore `simulation.db` tables and summary CSVs.
4. Load trajectory SQLite files for each mode.
5. Animate trajectories with risk overlays using your `animation.py`.
6. Visualize static trajectory plots, density plots, and risk snapshots.

It is designed around the current refactored pipeline:
- `run_from_yaml(...)` launches the simulation
- results are stored in a run directory under `runs/...`
- the main database is `artifacts/db/simulation.db`
- trajectories are stored as `*_mode_*.sqlite`

In [ ]:

from pathlib import Path
import sys
import sqlite3
import json
import shutil
import importlib.util

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, HTML
from jupedsim.internal.notebook_utils import read_sqlite_file

# --- Project root ---
PROJECT_ROOT = Path.cwd().parent.resolve()

# If the notebook is not placed at repo root, edit this manually:
# PROJECT_ROOT = Path("/absolute/path/to/Evacuation_Simulation").resolve()

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("PROJECT_ROOT =", PROJECT_ROOT)
print("SRC_DIR      =", SRC_DIR)

In [ ]:

# --- Load project functions ---
from evac_sim.runner import run_from_yaml
from evac_sim.db.sqlite_utils import read_trajectory_dataframe
from evac_sim.db.repositories.risk import get_risk_levels_by_frame
from evac_sim.io.run_paths import RUNS_DIRNAME
from evac_sim.viz.animation import animate

In [ ]:

# --- Discover available configs ---
CONFIGS_DIR = PROJECT_ROOT / "configs"
if not CONFIGS_DIR.exists():
    raise FileNotFoundError(f"Configs directory not found: {CONFIGS_DIR}")

available_configs = sorted(
    p.name for p in CONFIGS_DIR.glob("*.y*ml")
    if p.name != "defaults.yaml"
)

print("Available configs:")
for name in available_configs:
    print(" -", name)

## Simulation selection

Set the configuration you want to run here.

- `CONFIG_NAME`: YAML file inside `configs/`
- `CASE_ID`: specific case key inside that YAML
- `ENVIRONMENT`: optional environment filter
- `VERBOSE`: enables more detailed logging
- `OUT_DIR`: optional fixed output directory. Leave as `None` to create a new run automatically

In [ ]:

CONFIG_NAME = "management_building.yaml"
CASE_ID = "basement"    # e.g. "case_001"
ENVIRONMENT = None      # e.g. "building_a"
VERBOSE = True
OUT_DIR = None          # e.g. PROJECT_ROOT / "runs" / "debug_manual_run"

print("CONFIG_NAME =", CONFIG_NAME)
print("CASE_ID     =", CASE_ID)
print("ENVIRONMENT =", ENVIRONMENT)
print("VERBOSE     =", VERBOSE)
print("OUT_DIR     =", OUT_DIR)

## Run the simulation

Execute the next cell to launch the selected simulation.

The helper below detects the newly created run directory automatically.

In [ ]:

def list_run_dirs(project_root: Path) -> set[Path]:
    runs_dir = project_root / RUNS_DIRNAME
    if not runs_dir.exists():
        return set()
    return {p.resolve() for p in runs_dir.iterdir() if p.is_dir()}

before_runs = list_run_dirs(PROJECT_ROOT)

run_from_yaml(
    project_root=PROJECT_ROOT,
    config_name=CONFIG_NAME,
    case_id=CASE_ID,
    environment=ENVIRONMENT,
    out_dir=OUT_DIR,
    verbose=VERBOSE,
)

after_runs = list_run_dirs(PROJECT_ROOT)
new_runs = sorted(after_runs - before_runs)

if OUT_DIR is not None:
    RUN_DIR = Path(OUT_DIR).resolve()
elif len(new_runs) == 1:
    RUN_DIR = new_runs[0]
elif len(new_runs) > 1:
    RUN_DIR = new_runs[-1]
else:
    # fallback: last modified run directory
    RUN_DIR = max(after_runs, key=lambda p: p.stat().st_mtime)

ARTIFACTS_DIR = RUN_DIR / "artifacts"
DB_DIR = ARTIFACTS_DIR / "db"
CSV_DIR = ARTIFACTS_DIR / "csv"
IMAGES_DIR = ARTIFACTS_DIR / "images"
SIM_DB = DB_DIR / "simulation.db"

print("RUN_DIR      =", RUN_DIR)
print("ARTIFACTS    =", ARTIFACTS_DIR)
print("DB_DIR       =", DB_DIR)
print("CSV_DIR      =", CSV_DIR)
print("IMAGES_DIR   =", IMAGES_DIR)
print("SIM_DB       =", SIM_DB)

## Inspect the run directory

In [ ]:

def tree(path: Path, max_depth: int = 3, prefix: str = ""):
    if max_depth < 0 or not path.exists():
        return
    entries = sorted(path.iterdir(), key=lambda p: (p.is_file(), p.name.lower()))
    for i, entry in enumerate(entries):
        connector = "└── " if i == len(entries) - 1 else "├── "
        print(prefix + connector + entry.name)
        if entry.is_dir():
            extension = "    " if i == len(entries) - 1 else "│   "
            tree(entry, max_depth - 1, prefix + extension)

print(RUN_DIR.name)
tree(RUN_DIR, max_depth=4)

## Database inspection

In [ ]:

if not SIM_DB.exists():
    raise FileNotFoundError(f"simulation.db not found: {SIM_DB}")

conn = sqlite3.connect(SIM_DB)
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name",
    conn,
)
display(tables)

In [ ]:

def read_table(name: str, limit: int = 20) -> pd.DataFrame:
    return pd.read_sql_query(f"SELECT * FROM {name} LIMIT {limit}", conn)

for table_name in tables["name"].tolist():
    print(f"\n=== {table_name} ===")
    try:
        display(read_table(table_name, limit=10))
    except Exception as exc:
        print(f"Could not read {table_name}: {exc}")

## Summary CSVs

In [ ]:

csv_files = sorted(CSV_DIR.glob("*.csv"))
print("CSV files:")
for f in csv_files:
    print(" -", f.name)

csv_frames = {}
for f in csv_files:
    try:
        csv_frames[f.name] = pd.read_csv(f)
    except Exception as exc:
        print(f"Could not read {f.name}: {exc}")

for name, df in csv_frames.items():
    print(f"\n=== {name} ===")
    display(df.head(20))

## Trajectory files by mode

In [ ]:

trajectory_files = sorted(DB_DIR.glob("*_mode_*.sqlite"))
if not trajectory_files:
    raise FileNotFoundError(f"No trajectory SQLite files found in {DB_DIR}")

mode_to_trajectory = {}
for f in trajectory_files:
    stem = f.stem
    # expected pattern: <env_name>_mode_<mode>
    try:
        mode = int(stem.split("_mode_")[-1])
        mode_to_trajectory[mode] = f
    except Exception:
        print("Could not infer mode from:", f.name)

print("Detected trajectory files:")
for mode, file in sorted(mode_to_trajectory.items()):
    print(f" - mode {mode}: {file.name}")

In [ ]:

trajectory_dataframes = {}
trajectory_fps = {}

for mode, traj_file in sorted(mode_to_trajectory.items()):
    df, fps = read_trajectory_dataframe(traj_file)
    trajectory_dataframes[mode] = df
    trajectory_fps[mode] = fps
    print(f"Mode {mode}: rows={len(df):,}, fps={fps}")
    display(df.head(10))

## Read risks from `simulation.db`

This cell tries to infer the `case_name` used in `risk_data`.
If you already know it, set `CASE_NAME_IN_DB` manually.

In [ ]:

risk_table_exists = "risk_data" in set(tables["name"])
if risk_table_exists:
    risk_case_names = pd.read_sql_query(
        "SELECT DISTINCT case_name FROM risk_data ORDER BY case_name",
        conn,
    )
    display(risk_case_names)
else:
    risk_case_names = pd.DataFrame(columns=["case_name"])

CASE_NAME_IN_DB = None
if not risk_case_names.empty:
    CASE_NAME_IN_DB = risk_case_names.iloc[0]["case_name"]

print("CASE_NAME_IN_DB =", CASE_NAME_IN_DB)

In [ ]:

all_risks = {}

if risk_table_exists and CASE_NAME_IN_DB is not None:
    risk_df = pd.read_sql_query(
        '''
        SELECT case_name, frame, area, risk_level
        FROM risk_data
        WHERE case_name = ?
        ORDER BY frame, area
        ''',
        conn,
        params=(CASE_NAME_IN_DB,),
    )
    display(risk_df.head(20))

    grouped = risk_df.groupby("frame")
    all_risks = {
        int(frame): dict(zip(group["area"], group["risk_level"]))
        for frame, group in grouped
    }

    print("Loaded risk frames:", len(all_risks))
else:
    print("No risk_data table or case_name available. all_risks will stay empty.")

## Static trajectory plots by mode

In [ ]:

def plot_trajectories(mode: int, alpha: float = 0.3, linewidth: float = 0.7):
    df = trajectory_dataframes[mode]
    plt.figure(figsize=(9, 7))
    if df.empty:
        plt.title(f"Mode {mode} - empty trajectory data")
        plt.show()
        return

    for _, g in df.sort_values(["id", "frame"]).groupby("id"):
        plt.plot(g["pos_x"], g["pos_y"], alpha=alpha, linewidth=linewidth)

    plt.title(f"Trajectories - mode {mode}")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.axis("equal")
    plt.grid(True, alpha=0.2)
    plt.show()

for mode in sorted(trajectory_dataframes):
    plot_trajectories(mode)

## Display generated PNG artifacts

In [ ]:
from IPython.display import display, HTML
import matplotlib.image as mpimg

png_files = sorted(IMAGES_DIR.glob("*.png"))
print("Image artifacts:")
for f in png_files:
    print(" -", f.name)

for f in png_files:
    display(HTML(f"<h4>{f.name}</h4>"))

    img = mpimg.imread(f)
    plt.figure(figsize=(12, 8))  # ancho, alto en pulgadas
    plt.imshow(img)
    plt.axis("off")
    plt.show()

## Animation with your `animation.py`

This reproduces the style you wanted:
- read each trajectory SQLite file
- load the walkable area
- animate each mode
- overlay risks by frame
- optionally remove obstacles from `specific_areas`

You may need to adapt the environment-loading cell below if you want exact `specific_areas`
for a particular case/environment.

In [ ]:

# --- Optional: load environment-specific polygons for risk overlays ---
# Adapt these values if needed.
#
# If your config/case already implies an environment and you want exact overlays,
# set ENV_NAME_FOR_OVERLAY and implement the environment loading logic below.

ENV_NAME_FOR_OVERLAY = None
specific_areas_overlay = None

try:
    import evac_sim.envs.environment as pol
    from evac_sim.envs.environment_factory import select_environment

    if ENVIRONMENT is not None:
        ENV_NAME_FOR_OVERLAY = ENVIRONMENT
    else:
        # fallback: try to infer from metadata.json if available
        metadata_file = RUN_DIR / "metadata.json"
        if metadata_file.exists():
            metadata = json.loads(metadata_file.read_text(encoding="utf-8"))
            ENV_NAME_FOR_OVERLAY = metadata.get("environment")

    if ENV_NAME_FOR_OVERLAY:
        env = select_environment(ENV_NAME_FOR_OVERLAY)
        specific_areas_overlay = pol.remove_obstacles_from_areas(
            env.specific_areas,
            env.obstacles,
        )
        print("Loaded specific_areas for overlay from environment:", ENV_NAME_FOR_OVERLAY)
    else:
        print("No environment selected for specific area overlay. Animation will run without colored areas.")
except Exception as exc:
    print("Could not prepare specific_areas overlay:", exc)
    specific_areas_overlay = None

In [ ]:

mode_names = {
    0: "Efficient / Low awareness",
    1: "Efficient / High awareness",
    2: "Centrality / Low awareness",
    3: "Centrality / High awareness",
}

for mode in sorted(mode_to_trajectory):
    trajectory_file = mode_to_trajectory[mode]
    traj_data, walkable_area = read_sqlite_file(str(trajectory_file))

    fig = animate(
        traj_data,
        walkable_area,
        title_note=f"Mode: {mode_names.get(mode, str(mode))}",
        risk_per_frame=all_risks if all_risks else None,
        specific_areas=specific_areas_overlay,
        every_nth_frame=50,
    )
    fig.show()

## Optional: animate only one selected mode

In [ ]:

SELECTED_MODE = sorted(mode_to_trajectory)[0]
ANIMATION_EVERY_NTH_FRAME = 50

traj_data, walkable_area = read_sqlite_file(str(mode_to_trajectory[SELECTED_MODE]))

fig = animate(
    traj_data,
    walkable_area,
    title_note=f"Mode: {mode_names.get(SELECTED_MODE, str(SELECTED_MODE))}",
    risk_per_frame=all_risks if all_risks else None,
    specific_areas=specific_areas_overlay,
    every_nth_frame=ANIMATION_EVERY_NTH_FRAME,
)
fig.show()

## Close the database connection when finished

In [ ]:

conn.close()
print("Closed database connection.")